# 04 — Silver → Gold

Constroi o **modelo dimensional** (star schema) na camada **Gold**: dimensoes de cliente e produto e a fato de pedidos.

In [ ]:
from pyspark.sql.functions import (
    col,
    when,
    current_timestamp,
    year,
    month,
    dayofmonth
)

SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
""")

In [ ]:
# DIM CLIENTE
dim_cliente = (
    spark.table(f"{SILVER_SCHEMA}.clientes")
        .select(
            col("id").alias("cliente_id"),
            col("nome").alias("nome_cliente"),
            col("email"),
            col("cidade")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_cliente.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_cliente")
)

In [ ]:
# DIM PRODUTO
dim_produto = (
    spark.table(f"{SILVER_SCHEMA}.produtos")
        .select(
            col("id").alias("produto_id"),
            col("nome").alias("nome_produto"),
            col("categoria"),
            col("preco")
        )
        .withColumn("gold_created_at", current_timestamp())
)

(
    dim_produto.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.dim_produto")
)

In [ ]:
# FATO PEDIDO
pedidos = spark.table(f"{SILVER_SCHEMA}.pedidos")

fato_pedido = (
    pedidos
        .select(
            col("id").alias("pedido_id"),
            col("cliente_id"),
            col("produto_id"),
            col("data_pedido"),
            year(col("data_pedido")).alias("ano"),
            month(col("data_pedido")).alias("mes"),
            dayofmonth(col("data_pedido")).alias("dia"),
            col("quantidade"),
            col("valor_total"),
            col("status"),
            current_timestamp().alias("gold_created_at")
        )
)

(
    fato_pedido.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{GOLD_SCHEMA}.fato_pedido")
)

In [ ]:
gold_tables = [
    "dim_cliente",
    "dim_produto",
    "fato_pedido"
]

print("Camada Gold criada com sucesso.\n")

for table in gold_tables:
    count = spark.table(f"{GOLD_SCHEMA}.{table}").count()
    print(f"{table}: {count} registros")